In [ ]:
import pickle

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction import DictVectorizer

from sklearn.metrics import roc_auc_score

import xgboost as xgb

In [ ]:
!nvidia-smi

In [ ]:
train_path = '/content/train.csv'
test_path = '/content/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

(15000, 20) 
 (10000, 19)


In [ ]:
train_df.drop("id", axis=1, inplace=True)
list_test_id = test_df["id"].copy().to_list()
test_df.drop("id", axis=1, inplace=True)

In [ ]:
y = train_df['Status']
X = train_df.drop(columns=['Status'])

In [ ]:
# !pip install catboost optuna

In [ ]:
num_features = ["N_Days", "Age", "Bilirubin", "Cholesterol", "Albumin", "Copper",
                "Alk_Phos", "SGOT", "Tryglicerides", "Platelets", "Prothrombin"]

for df in [train_df, test_df]:
    for col in num_features:
        df[f"{col}_isna"] = df[col].isna().astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(train_df["Status"])
train_df = train_df.drop(columns=["Status"])

In [ ]:
if "Stage" in train_df.columns:
    train_df["Stage"] = train_df["Stage"].astype(str)
    test_df["Stage"] = test_df["Stage"].astype(str)

cat_cols = train_df.select_dtypes(include="object").columns.tolist()
num_cols = train_df.select_dtypes(exclude="object").columns.tolist()

In [ ]:
for col in train_df.columns:
    if train_df[col].isna().sum() > 0:
        train_df[col + "_isna"] = train_df[col].isna().astype(int)
        test_df[col + "_isna"] = test_df[col].isna().astype(int)

In [ ]:
skewed = ["Alk_Phos", "Copper", "SGOT", "Platelets", "Tryglicerides"]

for col in skewed:
    train_df[col] = np.log1p(train_df[col])
    test_df[col] = np.log1p(test_df[col])

In [ ]:
MISSING = "__MISSING__"

for c in cat_cols:
    train_df[c] = train_df[c].fillna(MISSING).astype(str)
    test_df[c]  = test_df[c].fillna(MISSING).astype(str)

In [ ]:
eps = 1e-6
train_df["Plat_Prot"] = train_df["Platelets"] / (train_df["Prothrombin"] + eps)
test_df["Plat_Prot"]  = test_df["Platelets"] / (test_df["Prothrombin"] + eps)

In [ ]:
# !pip uninstall xgboost -y
# !pip install xgboost==2.0.3

In [ ]:
import optuna
import xgboost as xgb

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

In [ ]:
def objective(trial):
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": trial.suggest_int("iterations", 1800, 4500),
        "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.06, log=True),
        "depth": trial.suggest_int("depth", 4, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 8.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 64),
        "bootstrap_type": "Bayesian",
        # "auto_class_weights": trial.suggest_categorical("auto_class_weights", ["Balanced", None]),
        "random_seed": 42,
        "verbose": 200,
        "early_stopping_rounds": 300,
        "task_type": "GPU",
        "devices": "0",
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros((len(X), 3))

    for tr_idx, val_idx in skf.split(X, y_arr):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_cols,
            use_best_model=True,
        )
        oof[val_idx] = model.predict_proba(X_val)

    return log_loss(y_arr, oof)


In [ ]:
X = train_df.copy()
X_test = test_df.copy()
y_arr = y.copy()

for col in ["Drug", "Sex", "Ascites", "Hepatomegaly", "Spiders", "Edema", "Stage"]:
    if col in X.columns:
        X[col] = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)

cat_cols = X.select_dtypes(include="object").columns.tolist()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

best_params = study.best_params
best_params.update(
    {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "bootstrap_type": "Bayesian",
        "verbose": 200,
        "early_stopping_rounds": 300,
        "task_type": "GPU",
        "devices": "0",
    }
)

cat_seeds = [13, 42, 2024, 777]
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

cat_oof = np.zeros((len(X), 3))
cat_test = np.zeros((len(X_test), 3))

for seed in cat_seeds:
    seed_oof = np.zeros((len(X), 3))
    seed_test = np.zeros((len(X_test), 3))

    for tr_idx, val_idx in skf.split(X, y_arr):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        model = CatBoostClassifier(**best_params, random_seed=seed)
        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_cols,
            use_best_model=True,
        )

        seed_oof[val_idx] = model.predict_proba(X_val)
        seed_test += model.predict_proba(X_test) / skf.n_splits

    cat_oof += seed_oof / len(cat_seeds)
    cat_test += seed_test / len(cat_seeds)

print("лучший trial:", study.best_trial.number)
print("лучшая optuna CV:", study.best_value)
print("катбуст ансамбдь CV logloss:", log_loss(y_arr, cat_oof))

X_all = pd.concat([X, X_test], axis=0).reset_index(drop=True)
X_all = pd.get_dummies(X_all, columns=cat_cols, dummy_na=True)
X_all = X_all.astype(np.float32)

X_xgb = X_all.iloc[:len(X)].reset_index(drop=True)
X_test_xgb = X_all.iloc[len(X):].reset_index(drop=True)

xgb_params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": "gpu_hist",
    "predictor": "gpu_predictor",
    "learning_rate": 0.03,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.85,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.02,
    "reg_lambda": 2.0,
    "n_estimators": 4000,
}

xgb_seeds = [42, 2024, 777]
xgb_oof = np.zeros((len(X_xgb), 3))
xgb_test = np.zeros((len(X_test_xgb), 3))

for seed in xgb_seeds:
    seed_oof = np.zeros((len(X_xgb), 3))
    seed_test = np.zeros((len(X_test_xgb), 3))

    for tr_idx, val_idx in skf.split(X_xgb, y_arr):
        X_tr, X_val = X_xgb.iloc[tr_idx], X_xgb.iloc[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        model = xgb.XGBClassifier(
            **xgb_params,
            random_state=seed,
            early_stopping_rounds=200,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        seed_oof[val_idx] = model.predict_proba(X_val)
        seed_test += model.predict_proba(X_test_xgb) / skf.n_splits

    xgb_oof += seed_oof / len(xgb_seeds)
    xgb_test += seed_test / len(xgb_seeds)

print("XGBoost ансамбль CV logloss:", log_loss(y_arr, xgb_oof))

weights = np.linspace(0.0, 1.0, 101)
best_w, best_score = 1.0, 10.0
for w in weights:
    blend_oof = w * cat_oof + (1.0 - w) * xgb_oof
    score = log_loss(y_arr, blend_oof)
    if score < best_score:
        best_score = score
        best_w = w
test_preds = best_w * cat_test + (1.0 - best_w) * xgb_test

print(f"оптимальный вес модели катбуст в ансабле: {best_w:.2f}")
print(f"блендинг CV logloss: {best_score:.6f}")


[I 2026-02-23 05:40:09,181] A new study created in memory with name: no-name-eec22bd9-3500-49c0-a999-49ac93119f92


  0%|          | 0/30 [00:00<?, ?it/s]

0:	learn: 1.0468548	test: 1.0481429	best: 1.0481429 (0)	total: 213ms	remaining: 9m 55s
200:	learn: 0.3103414	test: 0.4082920	best: 0.4082920 (200)	total: 10.3s	remaining: 2m 13s
400:	learn: 0.2439458	test: 0.4058954	best: 0.4054189 (332)	total: 14.6s	remaining: 1m 26s
600:	learn: 0.1991548	test: 0.4090807	best: 0.4054189 (332)	total: 23.7s	remaining: 1m 26s
bestTest = 0.4054188639
bestIteration = 332
Shrink model to first 333 iterations.
0:	learn: 1.0472865	test: 1.0476270	best: 1.0476270 (0)	total: 23.9ms	remaining: 1m 6s
200:	learn: 0.3052829	test: 0.4098241	best: 0.4098241 (200)	total: 6.15s	remaining: 1m 19s
400:	learn: 0.2432608	test: 0.4065635	best: 0.4063250 (346)	total: 10.3s	remaining: 1m 1s
600:	learn: 0.1996697	test: 0.4078551	best: 0.4063250 (346)	total: 14.7s	remaining: 53.7s
bestTest = 0.4063250326
bestIteration = 346
Shrink model to first 347 iterations.
0:	learn: 1.0470724	test: 1.0471675	best: 1.0471675 (0)	total: 23.9ms	remaining: 1m 6s
200:	learn: 0.3112309	test: 0.3

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:160: UserWarning: [06:28:38] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:160: UserWarning: [06:28:38] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:160: UserWarning: [06:28:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:160: UserWarning: [06:28:43] WARNING: /workspace/src/common/error_ms

XGBoost ансамбль CV logloss: 0.3709439206024774
оптимальный вес модели катбуст в ансабле: 0.36
блендинг CV logloss: 0.369418


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: T